# 03 Commute Data Ingest and Clean

Fetch and clean commute/access data. FRED-hosted commute series are searched first. Census ACS is used as a controlled fallback for national trend and selected county snapshots.

## Method Notes

The ACS fallback uses `B08013_001E / B08012_001E` as directed for mean commute minutes. County snapshots are selected county proxies, not full metro areas. COVID and post-COVID years require interpretation caution.

In [ ]:
from pathlib import Path
import json, os, requests
import pandas as pd
import numpy as np
ROOT=Path.cwd()
if ROOT.name=='notebooks': ROOT=ROOT.parent
DATA_DIR=ROOT/'data'; RAW_DIR=DATA_DIR/'raw'; INTERIM_DIR=DATA_DIR/'interim'; PROCESSED_DIR=DATA_DIR/'processed'; REPORTS_DIR=ROOT/'reports'
for p in [RAW_DIR/'commute_fred', RAW_DIR/'census', INTERIM_DIR, PROCESSED_DIR, REPORTS_DIR]: p.mkdir(parents=True, exist_ok=True)
def load_local_env():
    env=ROOT/'.env'
    if env.exists():
        for line in env.read_text().splitlines():
            if line.strip() and not line.strip().startswith('#') and '=' in line:
                k,v=line.split('=',1); os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
try:
    from dotenv import load_dotenv; load_dotenv(ROOT/'.env')
except Exception: load_local_env()
def get_api_key(name, required=False):
    value=os.environ.get(name)
    if value: return value
    sp=ROOT/'.secrets'/'api_keys.json'
    if sp.exists():
        try:
            value=json.loads(sp.read_text()).get(name)
            if value: os.environ.setdefault(name,value); return value
        except json.JSONDecodeError: pass
    if required: raise RuntimeError(f'Missing {name}. Set it as an environment variable or in local .env.')
    return None
def write_csv(df,path):
    path=Path(path); path.parent.mkdir(parents=True, exist_ok=True); df.to_csv(path,index=False); print(f'wrote {path.relative_to(ROOT)} ({len(df):,} rows)')

In [ ]:
FRED_BASE='https://api.stlouisfed.org/fred'; FRED_API_KEY=get_api_key('FRED_API_KEY'); CENSUS_API_KEY=get_api_key('CENSUS_API_KEY', required=True)
def fred_observations(series_id):
    r=requests.get(f'{FRED_BASE}/series/observations', params={'series_id':series_id,'api_key':FRED_API_KEY,'file_type':'json'}, timeout=60)
    if r.status_code!=200: raise RuntimeError(f'FRED observations failed for {series_id}: HTTP {r.status_code}')
    df=pd.DataFrame(r.json().get('observations',[])); df['series_id']=series_id; df['date']=pd.to_datetime(df['date']); df['value']=pd.to_numeric(df['value'].replace('.',np.nan), errors='coerce')
    return df[['series_id','date','value','realtime_start','realtime_end']]
fred_frames=[]; cand_path=INTERIM_DIR/'fred_commute_candidate_series.csv'
if FRED_API_KEY and cand_path.exists():
    cands=pd.read_csv(cand_path); national=cands[cands.description.str.contains('United States', case=False, na=False)].head(10)
    if national.empty: national=cands.head(5)
    for _,row in national.iterrows():
        sid=row.series_id_or_dataset
        try:
            raw=fred_observations(sid); write_csv(raw, RAW_DIR/'commute_fred'/f'{sid}.csv')
            tmp=raw.copy(); tmp['year']=tmp.date.dt.year
            clean=tmp.groupby(['series_id','year'], as_index=False).agg(mean_commute_minutes=('value','mean'))
            clean['geography']='United States' if 'United States' in str(row.description) else str(row.description); clean['description']=row.description
            fred_frames.append(clean)
        except Exception as exc: print(f'FRED commute candidate failed: {sid}: {exc}')
commute_fred_clean=pd.concat(fred_frames, ignore_index=True) if fred_frames else pd.DataFrame(columns=['series_id','year','mean_commute_minutes','geography','description'])
write_csv(commute_fred_clean, INTERIM_DIR/'commute_fred_clean.csv')
def census_get(year,dataset,params,raw_name):
    url=f'https://api.census.gov/data/{year}/{dataset}'; params=dict(params); params['key']=CENSUS_API_KEY
    r=requests.get(url, params=params, timeout=60)
    (RAW_DIR/'census'/raw_name).write_text(json.dumps({'url':r.url.replace(CENSUS_API_KEY,'REDACTED'),'status_code':r.status_code,'text':r.text}, indent=2))
    if r.status_code!=200: raise RuntimeError(f'Census request failed {year} {dataset}: HTTP {r.status_code} {r.text[:160]}')
    payload=r.json()
    if len(payload)<2: raise RuntimeError(f'Census request returned no data for {year} {dataset}')
    return pd.DataFrame(payload[1:], columns=payload[0])
national=[]
for year in range(2005,2025):
    if year==2020: continue
    try:
        df=census_get(year,'acs/acs1',{'get':'NAME,B08013_001E,B08012_001E','for':'us:*'},f'acs1_us_commute_{year}.json')
        df['year']=year; df['geography']='United States'; national.append(df)
    except Exception as exc: print(f'National ACS 1-year commute unavailable for {year}: {exc}')
national_commute=pd.concat(national, ignore_index=True) if national else pd.DataFrame()
if not national_commute.empty:
    for col in ['B08013_001E','B08012_001E']: national_commute[col]=pd.to_numeric(national_commute[col], errors='coerce')
    national_commute['mean_commute_minutes']=national_commute.B08013_001E/national_commute.B08012_001E; national_commute['source']='Census ACS 1-year'; national_commute['geo_level']='national'
selected=[('Washington, DC area proxy','11','001'),('San Francisco area proxy','06','075'),('New York area proxy','36','061'),('Los Angeles area proxy','06','037'),('Atlanta area proxy','13','121'),('Dallas area proxy','48','113'),('Phoenix area proxy','04','013'),('Denver area proxy','08','031'),('Charlotte area proxy','37','119'),('Chicago area proxy','17','031')]
geo_rows=[]; snapshot_year=None
for year in [2023,2022,2021,2019]:
    trial=[]; failed=False
    for label,state,county in selected:
        try:
            df=census_get(year,'acs/acs5',{'get':'NAME,B08013_001E,B08012_001E,B25077_001E,B19013_001E','for':f'county:{county}','in':f'state:{state}'},f'acs5_selected_county_{state}_{county}_{year}.json')
            df['case_geography']=label; df['state_fips']=state; df['county_fips']=county; df['year']=year; trial.append(df)
        except Exception as exc:
            failed=True; print(f'Selected county ACS 5-year unavailable for {label} {year}: {exc}'); break
    if not failed and trial: geo_rows=trial; snapshot_year=year; break
geo_snapshot=pd.concat(geo_rows, ignore_index=True) if geo_rows else pd.DataFrame()
if not geo_snapshot.empty:
    for col in ['B08013_001E','B08012_001E','B25077_001E','B19013_001E']: geo_snapshot[col]=pd.to_numeric(geo_snapshot[col], errors='coerce')
    geo_snapshot['mean_commute_minutes']=geo_snapshot.B08013_001E/geo_snapshot.B08012_001E; geo_snapshot['median_home_value']=geo_snapshot.B25077_001E; geo_snapshot['median_household_income']=geo_snapshot.B19013_001E; geo_snapshot['home_value_to_income_ratio']=geo_snapshot.median_home_value/geo_snapshot.median_household_income; geo_snapshot['source']='Census ACS 5-year'; geo_snapshot['geo_level']='selected county proxy'
frames=[]
if not national_commute.empty: frames.append(national_commute[['source','geo_level','geography','NAME','year','B08013_001E','B08012_001E','mean_commute_minutes']])
if not geo_snapshot.empty:
    tmp=geo_snapshot[['source','geo_level','case_geography','NAME','year','B08013_001E','B08012_001E','mean_commute_minutes']].rename(columns={'case_geography':'geography'}); frames.append(tmp)
commute_census_clean=pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()
write_csv(commute_census_clean, INTERIM_DIR/'commute_census_clean.csv')
if not geo_snapshot.empty:
    write_csv(geo_snapshot[['year','case_geography','NAME','state_fips','county_fips','mean_commute_minutes','median_home_value','median_household_income','home_value_to_income_ratio','source','geo_level']], INTERIM_DIR/'commute_census_geo_snapshot.csv')
validation=[]
for label,df in [('fred_commute',commute_fred_clean),('census_commute',commute_census_clean)]:
    validation.append({'dataset':label,'rows':len(df),'geographies':int(df.geography.nunique()) if 'geography' in df and not df.empty else 0,'first_year':int(df.year.min()) if 'year' in df and not df.empty else None,'last_year':int(df.year.max()) if 'year' in df and not df.empty else None,'missing_mean_commute':int(df.mean_commute_minutes.isna().sum()) if 'mean_commute_minutes' in df else None,'zero_denominators':int((df.B08012_001E==0).sum()) if 'B08012_001E' in df and not df.empty else 0})
write_csv(pd.DataFrame(validation), INTERIM_DIR/'commute_validation_summary.csv')
commute_census_clean.head()